## 나라장터 공고 데이터 보정 프롬프트 가이드

본 노트북은 35건의 공고 데이터를 수동으로 보정하며 얻은 노하우를 바탕으로 작성되었습니다. 향후 AI 에이전트 자동화의 기초 설계도로 사용됩니다.

### 1. 검색 에이전트 프롬프트 (Search Agent)

- **목적:** 나라장터 상세 검색을 통해 정확한 공고 페이지에 접속하기 위한 지침
- **핵심 전략:** 공고기관명과 사업명을 조합하여 검색 정확도 향상

Role: 나라장터 입찰 데이터 수집 전문가 (G2B Data Scraper)

Context

당신은 국가종합전자조달시스템(나라장터)에서 특정 입찰 공고의 누락된 메타데이터를 찾아 보충하는 업무를 수행합니다.

Search Protocol (표준 검색 경로)

1. 접속: 나라장터 홈페이지 (g2b.go.kr)

2. 이동: 입찰정보 -> 용역 -> 공고현황 -> 상세검색

3. 입력:

    * [공고명]: 제공된 '나라장터 검색어' 입력

    * [공고연도]: 제공된 '검색 범위'의 연도 설정

4. 실행: 검색 버튼 클릭 후 해당 공고의 '공고번호' 클릭하여 상세페이지 진입

- Task: 데이터 추출 및 보정 (상세페이지에서 다음 항목을 찾아 정해진 형식으로 추출하십시오.)

    - 입찰참여 시작일: [YYYY-MM-DD HH:MM] 형식 (예: 2025-05-10 10:00)

    - 입찰참여 마감일: [YYYY-MM-DD HH:MM] 형식

    - 사업 금액(추정가격/예정가격): 숫자만 추출 (예: 150,000,000)

- Example Data (Few-shot)

    - 검색 대상: 강릉어선안전조업국 상황관제시스템 구축

    - 검색어: 수협중앙회 강릉어선안전조업국 상황관제시스템 구축

    - 보정 항목: 입찰 참여 시작일

- 출력 요구: 해당 공고의 '입찰집행/진행내역' 탭 또는 본문에서 '입찰서접수개시일시'를 찾아 기록할 것.

## 2. 데이터 추출 프롬프트 (Data Extraction)

- **목적:** 공고문 텍스트에서 입찰일, 예산 등 핵심 메타데이터 추출
- **핵심 전략:** 날짜 형식(YYYY-MM-DD) 통일 및 숫자 데이터 정제

Prompt:  
"아래는 나라장터에서 긁어온 공고문 텍스트야. 여기서 '입찰서 접수 시작일'과 '마감일', 그리고 '배정예산'을 찾아서 JSON 형식으로 출력해줘. 만약 금액에 콤마가 있으면 제거하고 숫자만 남겨줘. 날짜는 하이픈(-)으로 구분된 표준 형식이어야 해."

## 3. 수동 보정 시 발견된 예외 케이스 (Insights)

사례 1: 본문에 금액이 없을 경우 첨부파일 확인 필요

사례 2: 공고명만으로 검색되지 않을 시 공고기관 필터 추가 설정

### 3. 수동 보정 시 발견된 예외 케이스 및 처리 로직

수동 보정 과정에서 웹 UI의 표준 필드에 데이터가 누락되거나 왜곡된 경우, 아래의 예외 처리 규칙에 따라 데이터를 보정하였습니다.

| 예외 유형 | 발생 현상 및 원인 | 보정 규칙 (Logic) |
| --- | --- | --- |
| **직찰/집찰 공고** | 입찰 방법이 **직찰** 또는 **집찰**인 경우 시작/종료일 필드가 비어 있거나 **공고서 참조**로 표시됨 | **시작일**: 상세 일정의 **공고 게시** 일시를 적용
|||**종료일**: 상세 일정의 **개찰** 또는 **입찰서 제출 마감** 일시를 적용 |
| **기간 미표기 건** | 상세 검색 결과에는 날짜가 나오지 않으나 실제 입찰은 진행 중인 경우 | 상세 페이지 내부의 **입찰집행내역** 테이블에서 각 단계별 일시를 추출하여 정합성 확보 |
| **공고서 참조** | 시스템 필드 대신 첨부파일 내부에만 일정이 명시된 경우 | **공고서(PDF/HWP)** 본문 내의 입찰 일정 테이블을 참조하여 **제안서 제출 마감일**을 최종 종료일로 설정 |
| **사업비 우선순위** | 배정예산과 추정가격이 동시에 존재하거나 선택해야 하는 경우 | **배정예산(VAT 포함)**을 최우선으로 기록하여 실제 총 사업 규모를 반영 |
| **추정가격만 존재** | 배정예산 필드가 비어 있고 추정가격만 명시된 경우 | **추정가격**을 기록하되 비고란에 **VAT 제외 금액**임을 명시하여 데이터 왜곡 방지 |
| **사업비 비공개** | 시스템 필드에 예산이 **공고서 참조**로 표기되거나 비어 있음 | **Budget**: **0**으로 입력하여 수치형 데이터 유지
|||**Notes**: **사업비 비공개** 명시 후 공고문 내 예산 확인 시도 |
| **입찰한도액 표기** | 사업 금액 항목 대신 입찰한도액으로 명시된 경우 | **Budget**: 입찰한도액을 해당 사업의 예산(숫자)으로 기록 |
| **재공고 사업** | 유찰 등으로 동일 사업이 재공고된 경우 (공고번호 뒤에 -01, -02 등이 붙음) | **시작일**: 사업의 전체 이력 파악을 위해 **최초 공고일** 적용
|||**종료일**: 현재 유효한 **최종 재공고의 마감일** 적용 |
| **긴급 재공고** | 일정 촉박으로 긴급 재공고가 올라온 복합 케이스 | **번호**: 최신 번호 적용
|||**시작일**: 최초 공고 게시일 적용
|||**종료일**: 최신 재공고 마감일 적용 |
| **공고번호 형식 왜곡** | 11자리 이상의 숫자 또는 영문 혼합 번호가 엑셀에서 지수(E+) 형태로 변환됨 | **처리**: 셀 서식을 **텍스트(Text)**로 강제 지정하여 데이터 유실 및 뒷자리 0 변환 방지 |
| **검색어 불일치** | 사업명만으로 검색 시 유사 공고가 너무 많이 노출되는 경우 | **공고기관명 + 사업명** 조합으로 검색어를 재구성하여 정확한 타겟 공고 식별 |

**비고**

- 위 로직은 향후 AI 에이전트가 상세 페이지의 텍스트 뭉치(Unstructured Text)에서 날짜를 유추할 때 최우선 순위 가이드라인으로 활용함.
- 재공고 건의 경우 최초 공고일과 최종 마감일 사이의 간격을 통해 사업의 긴급도나 유찰 빈도를 분석할 수 있는 데이터로 활용 가능함.
- 사업비 산정 시 **배정예산(총액)**을 기준으로 통일하여, 추후 타 공공 데이터와의 결합 및 예산 규모별 필터링 시 정합성을 확보함.

**주의 및 조치 사항**

- **사업비 0 처리**: 예산이 비공개인 데이터는 분석 시 통계 왜곡을 방지하기 위해 별도의 예외 처리가 필요함.
- **데이터 무결성 확보**: 11자리 이상의 공고번호는 지수 형태로 저장되는 순간 뒷자리가 유실되어 복구가 불가능함. 반드시 엑셀 셀 왼쪽 상단에 **초록색 삼각형**이 보이는 **텍스트 상태**로 저장해야 함.
- **파이프라인 연동**: CSV 파일을 파이썬(Pandas)으로 로드할 때 해당 컬럼을 **문자열(str)** 타입으로 읽어오도록 설정하여 타입 불일치 및 지수 표기 에러를 원천 차단함.